# Advanced Boolean Retrieval and Positional Indexing 
Si implementa il sistema di Retrieval introdotto nel notebook precedente ma in modo più vicino alle scelte adottate nei sistemi reali.

Per prima cosa si fa la seguente, fondamentale, assunzione: una persona di solito non effettuerà mai query del tipo "NOT term" o "term1 OR NOT term2", in quanto il risultato di queste query è eccessivamente ampio e poco significativo. Per questo motivo si costruirà un motore che non suppora queste query.

Tale scelta ha molti vantaggi: rende il sistema più efficiente, semplifica il query processing, evita di costruire insiemi enormi e riflette comunque un uso realistico di querying da parte dell'utente.

Il sistema gestirà l'ordine degli operatori secondo parentesi, e in caso di assenza si segue la seguente struttura: OR separa le clausole a livello esterno, e all'interno di ogni clausola si ha una serie di termini connessi da AND. NOT può comparire solo come negazione locale all'interno di una clausola AND -> **una query è sempre trattata come OR di clausole AND**

Inoltre si permette anche la ricerca di frasi esatte, a patto che siano racchiuse tra virgolette -> phrase queries.

## Inverted Index, Positional Index e DF
Dopo aver effettuato il preprocessing ed aver salvato i documenti processati in un dict tokenized documents (dove chiave è doc_id e valore è la lista di token), si costruisce l'inverted index e il positional index.

```python
def build_inverted_index(tokenized_documents):
    """
    Costruisce un inverted index non posizionale.

    Struttura restituita
    --------------------
    dict[str, list[int]]
        termine -> posting list ordinata dei docID

    Idea
    ----
    In un indice non posizionale, se un termine compare più volte
    nello stesso documento, il docID deve comparire una sola volta
    nella sua posting list.

    Per questo, mentre scorriamo i token di un documento, teniamo
    traccia dei termini già visti nel documento corrente.
    """
    inverted_index = {}

    for doc_id, tokens in tokenized_documents.items():
        seen_terms = set()

        for token in tokens:
            # Se abbiamo già visto questo termine nello stesso documento,
            # non aggiungiamo di nuovo lo stesso docID.
            if token in seen_terms:
                continue

            seen_terms.add(token)

            if token not in inverted_index:
                inverted_index[token] = []

            inverted_index[token].append(doc_id)

    # Non facciamo una sort finale delle posting lists:
    # i documenti vengono già visitati in ordine crescente di docID,
    # quindi le posting lists risultano ordinate per costruzione.
    return inverted_index
```

(fondamentale il fatto che si eviti il sorting finale delle posting lists per costruzione)

```python
def build_positional_index(tokenized_documents):
    """
    Costruisce un positional index.

    Struttura restituita
    --------------------
    dict[str, dict[int, list[int]]]
        termine -> {docID -> lista delle posizioni}

    Esempio
    -------
    {
        "caesar": {
            1: [2],
            2: [0],
            4: [5]
        }
    }

    Nota
    ----
    Qui NON eliminiamo i duplicati del termine nello stesso documento,
    perché ogni occorrenza deve essere registrata con la sua posizione.
    """
    positional_index = {}

    for doc_id, tokens in tokenized_documents.items():
        for position, token in enumerate(tokens):
            if token not in positional_index:
                positional_index[token] = {}

            if doc_id not in positional_index[token]:
                positional_index[token][doc_id] = []

            positional_index[token][doc_id].append(position)

    return positional_index
```

Nota come nel positional index le posizioni non sono quelle del testo originale perché tokenized_documents è un dict del tipo doc_id -> lista di token del documento; però dal momento che il testo della query sarà tokenizzato allo stesso modo non ci saranno problemi di allineamento tra posizioni del positional index e posizioni dei token della query.

Nota però che ciò potrebbe essere problematico: file is format diventa file format, le frasi diventano indistinguibili. Quindi una phrase query "file format" potrebbe matchare entrambe.

Di seguito il calcolo della df per ciascun termine, si restituisce un dict del tipo term -> df(term) che è semplicemente la lunghezza della posting list di ciascun termine.

```python
def build_document_frequency(inverted_index):
    """
    Calcola la document frequency di ciascun termine.

    La document frequency di un termine è il numero di documenti
    in cui il termine compare almeno una volta.

    Parameters
    ----------
    inverted_index : dict[str, list[int]]
        Indice inverso non posizionale.

    Returns
    -------
    dict[str, int]
        termine -> document frequency
    """
    document_frequency = {}

    for term, posting_list in inverted_index.items():
        document_frequency[term] = len(posting_list)

    return document_frequency
```

## Merge, Phrase Queries e Query Processing
Partiamo dall'implementazione efficiente del merge per AND, OR e NOT AND. 

Con AND si scorre di volta in volta la posting list con docID più piccolo e si confronta con l'altro docID, se sono uguali si aggiunge alla lista risultato, altrimenti si avanza nella posting list che ha il docID più piccolo. Con OR si aggiunge ogni volta il docID più piccolo alla lista risultato, e si avanza nella posting list corrispondente. 

``` python
def intersect_sorted(a, b):
    i, j = 0, 0
    result = []

    while i < len(a) and j < len(b):
        if a[i] == b[j]:
            result.append(a[i])
            i += 1
            j += 1

        elif a[i] < b[j]:
            i += 1
        else:
            j += 1

    return result


def union_sorted(a, b):
    i, j = 0, 0
    result = []
    while i < len(a) and j < len(b):
        if a[i] == b[j]:
            result.append(a[i])
            i += 1
            j += 1

        elif a[i] < b[j]:
            result.append(a[i])
            i += 1
        else:
            result.append(b[j])
            j += 1
            
    # Se una delle due liste non è finita,
    # i suoi elementi rimanenti vanno tutti nell'unione.
    while i < len(a):
        result.append(a[i])
        i += 1

    while j < len(b):
        result.append(b[j])
        j += 1

    return result
```

Per quel che riguarda l'AND NOT, l'idea è semplice: si scorre sempre il docID più piccolo, se è uguale a quello dell'altra lista si avanza in entrambe senza aggiungere il docID al risultato, altrimenti si aggiunge alla lista risultato e si avanza. 

Nota che stiamo consideranto A AND NOT B, quindi nella parte finale solo se avanza la lista A si aggiungono i docID alla lista risultato, se invece avanza la lista B non si aggiunge nulla alla lista risultato.

```python
def difference_sorted(a, b):
    i, j = 0, 0
    result = []

    while i < len(a) and j < len(b):
        if a[i] == b[j]:
            i += 1
            j += 1

        elif a[i] < b[j]:
            result.append(a[i])
            i += 1
        else:
            j += 1
            
    # Se la prima lista non è finita, i suoi elementi rimanenti vanno tutti nella differenza.
    while i < len(a):
        result.append(a[i])
        i += 1

    return result
```

Di seguito per la ricerca di phrase queries. Si procede come segue: 
1. si trovano i documenti candidati, cioè quelli che contengono tutti i termini della frase (AND sui termini)
2. si verifica che in almeno uno di questi documenti le posizioni siano allineate in modo corretto

Esempio: per la frase ["file", "format"] dobbiamo trovare almeno una posizione p tale che: 
- "file" compaia in p
- "format" compaia in p + 1

```python
def phrase_query_docs(phrase_tokens, positional_index):
    """
    Restituisce i docID dei documenti che contengono esattamente la frase.

    Parameters
    ----------
    phrase_tokens : list[str]
        Lista dei token della frase, già preprocessati.
        Esempio: ["file", "format"]

    positional_index : dict
        Indice posizionale nel formato:
            termine -> {doc_id: [posizioni]}

        Esempio:
            "file"   -> {3: [7, 20], 8: [2]}
            "format" -> {3: [8], 8: [10]}

    Returns
    -------
    list[int]
        Lista ordinata dei docID che contengono la frase esatta.
    """
    if len(phrase_tokens) == 0:
        # Caso limite: frase vuota.
        # Non ha senso restituire documenti.
        return []

    if len(phrase_tokens) == 1:
        # Una frase di un solo termine coincide semplicemente
        # con la posting list di quel termine.
        #
        # Usiamo il positional index e prendiamo solo i docID.
        # I docID sono già ordinati per costruzione.
        return list(positional_index.get(phrase_tokens[0], {}).keys())

    postings_per_term = []

    for term in phrase_tokens:
        postings_for_term = positional_index.get(term, {})

        # Se anche uno solo dei termini non compare nell'indice,
        # la frase non può comparire in nessun documento.
        if len(postings_for_term) == 0:
            return []

        postings_per_term.append(postings_for_term)

    # Fase 1: identificazione dei documenti candidati
    common_docs = list(postings_per_term[0].keys())

    for postings_for_term in postings_per_term[1:]:
        common_docs = intersect_sorted(common_docs, list(postings_for_term.keys()))

    result_docs = []

    # Fase 2: verifica dell'allineamento delle posizioni nei documenti candidati
    for doc_id in common_docs:
        # Posizioni del primo termine nel documento corrente.
        #
        # Esempio:
        #   "file" -> [3, 10, 25]
        #
        # Ognuna di queste posizioni può essere un possibile inizio
        # della frase.
        first_positions = postings_per_term[0][doc_id]

        # Per i termini successivi convertiamo le liste di posizioni in set.
        #
        # Perché?
        # Perché poi dovremo fare molti controlli del tipo:
        #   "la posizione start_pos + 1 esiste?"
        #   "la posizione start_pos + 2 esiste?"
        #
        # Su un set questo test è molto rapido.
        later_position_sets = [set(postings[doc_id]) for postings in postings_per_term[1:]]

        # Flag che indica se abbiamo trovato almeno una occorrenza valida
        # della frase nel documento corrente.
        found = False

        # Proviamo ogni posizione del primo termine come possibile inizio.
        for start_pos in first_positions:
            match = True

            # Se la frase inizia in start_pos, allora:
            # - il secondo termine deve stare in start_pos + 1
            # - il terzo termine deve stare in start_pos + 2
            # - ...
            for offset, pos_set in enumerate(later_position_sets, start=1):
                if start_pos + offset not in pos_set:
                    # Appena un termine manca nella posizione attesa,
                    # questa partenza non è valida.
                    match = False
                    break

            if match:
                # Abbiamo trovato almeno una occorrenza corretta
                # della frase nel documento.
                found = True
                break

        if found:
            result_docs.append(doc_id)

    return result_docs
```

Dopodiché nel notebook si introduce un parser per le query booleane del tipo OR di clausole AND. Utile per implementazione ma inutile da imparare a memoria, recuperalo se servisse per progetto.

Limiti della versione attuale:
- non supporta query del tipo "NOT term" o "term1 OR NOT term2"
- non supporta operatori di prossimità per phrase queries (es. "file format"~5)
- non implementa grammatica booleana totalmente arbitraria (solo OR di clausole AND)
- non introduce ranking o ordinamento per rilevanza, ma questo perché di per sé il Boolean Retrieval non prevede ranking, è un modello di retrieval binario (relevant vs non relevant)